# PESO replication: does a continual-LoRA regularizer beat plain continual LoRA?

PESO (Yoo et al., ICLR 2026, [arXiv 2510.25093](https://arxiv.org/abs/2510.25093)) continually updates a Llama-3.2-1B generative recommender one chronological block at a time. It uses one LoRA adapter plus a penalty that keeps the adapter close to the previous block's. We reran its released Instruments setup with three seeds. This notebook reads only the **committed metric files** in `replication/peso/results/`; it trains and evaluates nothing.

These are results on PESO's released Instruments blocks, not the project's Electronics/Movies & TV data. Protocol, the two environment deviations (Python 3.10; one GPU with gradient accumulation 4) and caveats are in [`docs/PESO-REPLICATION.md`](../docs/PESO-REPLICATION.md).

In [ ]:
import importlib.util
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
spec = importlib.util.spec_from_file_location("summarize", root / "replication/peso/summarize.py")
summarize = importlib.util.module_from_spec(spec)
spec.loader.exec_module(summarize)

runs = [root / "replication/peso/results" / f"seed{s}" for s in (42, 43, 44)]
table = summarize.summarize(runs)
print(summarize.markdown(table))

## Table 2 against the paper

Each cell is the mean ± sd across seeds, with the paper's value in brackets. The paper averages D2–D4 (code blocks 1–3). Pretrain and single evolving LoRA land within about 5% of the paper. PESO's NDCG@10 is about 10% lower, so the paper's ordering single LoRA < PESO does not reproduce.

## Per block

Block 0 is the initial-training block. PESO and single LoRA share each seed's block-0 checkpoint, so their block-0 scores must be identical, which serves as a sanity check.

In [ ]:
metric = "ndcg@10"
print(f"{metric} by test block (each method is evaluated on block j after training through block j)")
print("| Seed | Method | " + " | ".join(f"b{j}" for j in range(5)) + " |")
print("| --- | --- |" + " --- |" * 5)
for seed, run in zip((42, 43, 44), runs):
    for method, tag in summarize.TAGS.items():
        blocks = summarize.per_block(run / "results" / tag)
        print(f"| {seed} | {method} | " + " | ".join(f"{blocks[j][metric]:.4f}" for j in range(5)) + " |")

## Paired difference: PESO − single evolving LoRA

Both methods start from the same block-0 checkpoint and see the same data in each seed, so their per-seed difference is far less noisy than either average.

In [ ]:
diff = summarize.paired(runs, "lora", "peso")
print("| Metric | per seed (42, 43, 44) | mean ± sd | paper |")
print("| --- | --- | --- | --- |")
for m, d in diff.items():
    per_run = ", ".join(f"{x:+.4f}" for x in d["per_run"])
    print(f"| {m} | {per_run} | {d['mean']:+.4f} ± {d['sd']:.4f} | {d['paper']:+.4f} |")

## How large is seed noise?

Single LoRA's own improvement over Pretrain changes sign across seeds. A single-seed table, as in the paper, cannot separate a gain of about 0.001 from this variation.

In [ ]:
noise = summarize.paired(runs, "pretrain", "lora")
for m in ("ndcg@10", "hit@10"):
    d = noise[m]
    print(f"single LoRA - Pretrain, {m}: " + ", ".join(f"{x:+.4f}" for x in d["per_run"]) + f"  (paper {d['paper']:+.4f})")

## Next: transfer matrix and per-user flips (not run yet)

Table 2 only scores each block right after training on it. The planned analysis evaluates every saved block state on every block's users, using `replication/peso/peso_eval.py` on the GPU pod. It then reports backward transfer and per-user kept/lost/gained counts with `cp4285 flips`. Once those prediction dumps exist, the cell below reads them; until then it says so.

In [ ]:
from cp4285.experiment import flips

dumps = root / "runs/peso/seed42/predictions/peso"
if any(dumps.glob("state*_block*.jsonl")):
    report = flips.report(dumps, k=10)
    print("backward transfer (NDCG@10):", report["backward_transfer"])
else:
    print(f"No prediction dumps at {dumps.relative_to(root)} yet; run the matrix stage first.")